# EcoSignal Active Sampling Planner v1 — Real Map Data

This notebook uses the same **real GPS-aligned inputs** as `GeoRisk3D`:

- `Dem_2015.tif`
- `Dem_2022.tif`
- `DemDif_2015-2022.tif`
- `lichen_points.csv`

It rebuilds the aligned EcoSignal risk layers, generates candidate field locations from the real raster grid, ranks their scientific value, enforces geographic spacing, explains every recommendation, and exports CSV + GeoJSON files for the EcoSignal frontend.

No synthetic observations are generated or used.

> Scientific boundary: model uncertainty in v1 is an explicit **sampling-priority heuristic**, combining distance from observations, disagreement between geomorphology and lichen signals, and local CV confidence. It is not yet a statistically calibrated prediction interval.


In [ ]:
# Colab setup
import importlib.util, subprocess, sys

missing = [p for p in ("rasterio", "scipy", "pyproj") if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

import json, math, warnings
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from rasterio.windows import from_bounds, Window
from rasterio.warp import transform as warp_transform, transform_bounds
from scipy.ndimage import gaussian_filter, uniform_filter, distance_transform_edt
from IPython.display import display

pd.set_option('display.max_colwidth', 120)
print('Environment ready.')


## 1. Configuration

The defaults match the folder and filenames in your GeoRisk3D notebook. Planner weights are deliberately visible and must total 1.0.


In [ ]:
@dataclass(frozen=True)
class PlannerConfig:
    dem_dir: str = "/content/drive/MyDrive/EcoSignal_DEM"
    crop_buffer_m: float = 500.0
    dem_limit_of_detection_m: float = 0.15
    lichen_smoothing_sigma_px: float = 60.0
    candidate_spacing_m: float = 30.0
    exclude_near_sample_m: float = 30.0
    minimum_selected_spacing_m: float = 120.0
    distance_uncertainty_cap_m: float = 750.0
    recommendations: int = 10
    minimum_controls: int = 2
    minimum_hotspots: int = 3
    minimum_uncertainty_checks: int = 2
    risk_weight: float = 0.28
    uncertainty_weight: float = 0.24
    sparsity_weight: float = 0.18
    recent_change_weight: float = 0.15
    control_weight: float = 0.10
    terrain_diversity_weight: float = 0.05

    def validate(self):
        weights = [self.risk_weight, self.uncertainty_weight, self.sparsity_weight,
                   self.recent_change_weight, self.control_weight, self.terrain_diversity_weight]
        if not np.isclose(sum(weights), 1.0):
            raise ValueError(f"Planner weights must total 1.0; received {sum(weights):.4f}")
        if self.recommendations < 1 or self.minimum_selected_spacing_m <= 0:
            raise ValueError("Recommendation count and spacing must be positive.")
        return self

CONFIG = PlannerConfig().validate()
DEM_DIR = Path(CONFIG.dem_dir)
PATHS = {
    "dem_2015": DEM_DIR / "Dem_2015.tif",
    "dem_2022": DEM_DIR / "Dem_2022.tif",
    "dem_difference": DEM_DIR / "DemDif_2015-2022.tif",
    "lichen_points": DEM_DIR / "lichen_points.csv",
}

display(pd.Series(asdict(CONFIG), name='value').to_frame())
print('\nInputs:')
for name, path in PATHS.items():
    print(f"- {name}: {path}")


## 2. Load and validate the real inputs

The planner fails loudly if a file or required field is missing. It also requires a projected DEM in meters so distance and spacing rules remain scientifically meaningful.


In [ ]:
missing_files = [str(path) for path in PATHS.values() if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "Missing EcoSignal input files:\n- " + "\n- ".join(missing_files) +
        "\nUpload them to MyDrive/EcoSignal_DEM or change CONFIG.dem_dir."
    )

lichen_df = pd.read_csv(PATHS["lichen_points"])
required_columns = {
    "latitude", "longitude", "coverage_percent", "chlorosis_score",
    "nitrophyte_score", "growth_form", "cv_confidence"
}
missing_columns = required_columns - set(lichen_df.columns)
if missing_columns:
    raise ValueError(f"lichen_points.csv is missing columns: {sorted(missing_columns)}")

numeric_01 = ["chlorosis_score", "nitrophyte_score", "cv_confidence"]
for col in numeric_01:
    lichen_df[col] = pd.to_numeric(lichen_df[col], errors='coerce')
    if lichen_df[col].isna().any() or not lichen_df[col].between(0, 1).all():
        raise ValueError(f"{col} must contain numeric values from 0 to 1.")

lichen_df["coverage_percent"] = pd.to_numeric(lichen_df["coverage_percent"], errors='coerce')
if lichen_df["coverage_percent"].isna().any() or not lichen_df["coverage_percent"].between(0, 100).all():
    raise ValueError("coverage_percent must contain numeric values from 0 to 100.")
if not lichen_df.latitude.between(-90, 90).all() or not lichen_df.longitude.between(-180, 180).all():
    raise ValueError("Invalid latitude/longitude values in lichen_points.csv.")

with rasterio.open(PATHS["dem_2022"]) as src:
    if not src.crs or not src.crs.is_projected:
        raise ValueError("The DEM must use a projected CRS for meter-based distance calculations.")
    print("DEM CRS:", src.crs)
    print("DEM resolution:", src.res)

print(f"Validated {len(lichen_df)} real lichen observations.")
display(lichen_df.head())


## 3. Rebuild the GPS-aligned GeoRisk layers

This follows the real-data portion of GeoRisk3D: crop around the observations, confirm DEM-difference direction, compute slope and erosion risk, and create the confidence-weighted lichen-stress surface.


In [ ]:
def get_wgs84_bounds(df, buffer_m):
    min_lat, max_lat = df.latitude.min(), df.latitude.max()
    min_lon, max_lon = df.longitude.min(), df.longitude.max()
    mean_lat = np.deg2rad((min_lat + max_lat) / 2)
    lat_buffer = buffer_m / 111_320
    lon_buffer = buffer_m / (111_320 * np.cos(mean_lat))
    return min_lon-lon_buffer, min_lat-lat_buffer, max_lon+lon_buffer, max_lat+lat_buffer


def read_crop(path, bounds_wgs84):
    with rasterio.open(path) as src:
        raster_bounds = transform_bounds("EPSG:4326", src.crs, *bounds_wgs84, densify_pts=21)
        window = from_bounds(*raster_bounds, transform=src.transform).round_offsets().round_lengths()
        window = window.intersection(Window(0, 0, src.width, src.height))
        data = src.read(1, window=window, masked=True)
        transform = src.window_transform(window)
        profile = src.profile.copy()
        profile.update(height=data.shape[0], width=data.shape[1], transform=transform)
    return data, transform, profile


def normalize_percentile(arr, low=5, high=95):
    valid = np.isfinite(arr)
    if not valid.any():
        return np.zeros_like(arr, dtype='float32')
    p_low, p_high = np.nanpercentile(arr[valid], [low, high])
    if np.isclose(p_low, p_high):
        return np.zeros_like(arr, dtype='float32')
    return np.clip((arr-p_low)/(p_high-p_low), 0, 1).astype('float32')


crop_bounds_wgs84 = get_wgs84_bounds(lichen_df, CONFIG.crop_buffer_m)
dem_2015, transform_2015, profile_2015 = read_crop(PATHS["dem_2015"], crop_bounds_wgs84)
dem_2022, transform_2022, profile_2022 = read_crop(PATHS["dem_2022"], crop_bounds_wgs84)
dem_difference, transform_difference, profile_difference = read_crop(PATHS["dem_difference"], crop_bounds_wgs84)

if not (dem_2015.shape == dem_2022.shape == dem_difference.shape):
    raise ValueError(f"Cropped raster shapes do not align: {dem_2015.shape}, {dem_2022.shape}, {dem_difference.shape}")

d15 = dem_2015.filled(np.nan).astype('float32')
d22 = dem_2022.filled(np.nan).astype('float32')
dd_file = dem_difference.filled(np.nan).astype('float32')
dd_calc = d22 - d15
valid = np.isfinite(dd_calc) & np.isfinite(dd_file)
if valid.sum() < 2:
    raise ValueError("Not enough overlapping valid DEM pixels.")

diff_correlation = np.corrcoef(dd_calc[valid].ravel(), dd_file[valid].ravel())[0, 1]
change = -dd_file if diff_correlation < 0 else dd_file
clean_change = np.where(np.abs(change) < CONFIG.dem_limit_of_detection_m, 0, change)

xres, yres = abs(transform_2022.a), abs(transform_2022.e)
dz_dy, dz_dx = np.gradient(d22, yres, xres)
slope_2022 = np.degrees(np.arctan(np.sqrt(dz_dx**2 + dz_dy**2)))
slope_score = normalize_percentile(slope_2022)
past_erosion = np.where(clean_change < 0, np.abs(clean_change), 0)
past_erosion_risk = normalize_percentile(past_erosion)
geomorph_risk = np.clip(0.55*slope_score + 0.45*past_erosion_risk, 0, 1)

print("Crop shape:", d22.shape)
print("Difference correlation:", round(float(diff_correlation), 4))
print("Pixel size (m):", (xres, yres))


In [ ]:
def growth_form_stress(value):
    return {"crustose": 0.45, "foliose": 0.25, "fruticose": 0.15}.get(str(value).lower().strip(), 0.30)


def calculate_lichen_stress(row):
    coverage_stress = 1 - np.clip(row.coverage_percent / 80, 0, 1)
    stress = (0.35*row.chlorosis_score + 0.25*coverage_stress +
              0.25*row.nitrophyte_score + 0.15*growth_form_stress(row.growth_form))
    return float(np.clip(stress, 0, 1))


lichen_points = lichen_df.copy()
lichen_points["lichen_stress"] = lichen_points.apply(calculate_lichen_stress, axis=1)
xs, ys = warp_transform("EPSG:4326", profile_2022["crs"],
                        lichen_points.longitude.to_numpy(), lichen_points.latitude.to_numpy())
lichen_points["x"], lichen_points["y"] = xs, ys
pixel_pairs = [rasterio.transform.rowcol(transform_2022, x, y) for x, y in zip(xs, ys)]
lichen_points["row"] = [p[0] for p in pixel_pairs]
lichen_points["col"] = [p[1] for p in pixel_pairs]
height, width = d22.shape
lichen_points["inside_crop"] = (lichen_points.row.between(0, height-1) & lichen_points.col.between(0, width-1))
inside_points = lichen_points[lichen_points.inside_crop].copy()
if inside_points.empty:
    raise ValueError("No lichen observations fall inside the DEM crop.")

stress_grid = np.zeros(d22.shape, dtype='float32')
confidence_grid = np.zeros(d22.shape, dtype='float32')
presence_grid = np.zeros(d22.shape, dtype='float32')
for p in inside_points.itertuples():
    r, c = int(p.row), int(p.col)
    stress_grid[r, c] += float(p.lichen_stress) * float(p.cv_confidence)
    confidence_grid[r, c] += float(p.cv_confidence)
    presence_grid[r, c] += 1

smooth_stress = gaussian_filter(stress_grid, sigma=CONFIG.lichen_smoothing_sigma_px)
smooth_weight = gaussian_filter(confidence_grid, sigma=CONFIG.lichen_smoothing_sigma_px)
lichen_stress_surface = np.divide(smooth_stress, smooth_weight,
                                  out=np.zeros_like(smooth_stress), where=smooth_weight > 1e-12)
lichen_stress_surface = np.clip(lichen_stress_surface, 0, 1)

smooth_confidence = gaussian_filter(confidence_grid, sigma=CONFIG.lichen_smoothing_sigma_px)
smooth_presence = gaussian_filter(presence_grid, sigma=CONFIG.lichen_smoothing_sigma_px)
local_cv_confidence = np.divide(smooth_confidence, smooth_presence,
                                out=np.zeros_like(smooth_confidence), where=smooth_presence > 1e-12)
local_cv_confidence = np.clip(local_cv_confidence, 0, 1)

ecosignal_risk = np.clip(0.70*geomorph_risk + 0.30*lichen_stress_surface, 0, 1)
print(f"Mapped {len(inside_points)} real lichen observations into the aligned raster.")


## 4. Build real candidate sites and priority signals

Candidate points come directly from valid pixels of the real aligned raster. Terrain strata are derived from topographic position because this notebook does not yet contain a land-use/public-access layer. Consequently, every result is labeled `field_verification_required` rather than being presented as automatically accessible.


In [ ]:
# Topographic strata from the real 2022 DEM
elevation_norm = normalize_percentile(d22)
local_window_px = max(3, int(round(101 / np.mean([xres, yres]))))
if local_window_px % 2 == 0:
    local_window_px += 1
local_mean = uniform_filter(np.nan_to_num(d22, nan=float(np.nanmean(d22))), size=local_window_px)
tpi = d22 - local_mean
ridge_score = normalize_percentile(tpi)
valley_score = normalize_percentile(-tpi)

# Real-data uncertainty heuristic
sample_mask = np.zeros(d22.shape, dtype=bool)
sample_mask[inside_points.row.astype(int), inside_points.col.astype(int)] = True
distance_to_sample_m = distance_transform_edt(~sample_mask, sampling=(yres, xres))
distance_uncertainty = np.clip(distance_to_sample_m / CONFIG.distance_uncertainty_cap_m, 0, 1)
signal_disagreement = np.abs(geomorph_risk - lichen_stress_surface)
low_cv_confidence = 1 - local_cv_confidence
uncertainty_score = np.clip(0.45*distance_uncertainty + 0.35*signal_disagreement + 0.20*low_cv_confidence, 0, 1)

recent_change_score = normalize_percentile(np.abs(clean_change), low=50, high=98)
sparsity_score = distance_uncertainty.copy()
control_score = np.clip((0.35-ecosignal_risk)/0.35, 0, 1) * sparsity_score

candidate_step = max(1, int(round(CONFIG.candidate_spacing_m / np.mean([xres, yres]))))
rows = np.arange(0, height, candidate_step)
cols = np.arange(0, width, candidate_step)
rr, cc = np.meshgrid(rows, cols, indexing='ij')
rr, cc = rr.ravel(), cc.ravel()
valid_candidate = (np.isfinite(d22[rr, cc]) &
                   (distance_to_sample_m[rr, cc] >= CONFIG.exclude_near_sample_m))
rr, cc = rr[valid_candidate], cc[valid_candidate]

candidate_x, candidate_y = rasterio.transform.xy(transform_2022, rr, cc, offset='center')
candidate_lon, candidate_lat = warp_transform(profile_2022["crs"], "EPSG:4326", candidate_x, candidate_y)

candidates = pd.DataFrame({
    "row": rr, "col": cc, "x": candidate_x, "y": candidate_y,
    "longitude": candidate_lon, "latitude": candidate_lat,
    "risk_score": ecosignal_risk[rr, cc],
    "uncertainty_score": uncertainty_score[rr, cc],
    "sparsity_score": sparsity_score[rr, cc],
    "recent_change_score": recent_change_score[rr, cc],
    "control_score": control_score[rr, cc],
    "distance_to_nearest_sample_m": distance_to_sample_m[rr, cc],
    "slope_score": slope_score[rr, cc], "ridge_score": ridge_score[rr, cc],
    "valley_score": valley_score[rr, cc],
})

candidates["terrain_type"] = np.select(
    [candidates.valley_score >= 0.65,
     candidates.ridge_score >= 0.60,
     candidates.slope_score >= 0.45],
    ["lowland", "ridge", "slope"], default="other"
)

# Compare candidate terrain classes with the terrain classes already represented by observations.
sample_terrain = np.select(
    [valley_score[inside_points.row.astype(int), inside_points.col.astype(int)] >= 0.65,
     ridge_score[inside_points.row.astype(int), inside_points.col.astype(int)] >= 0.60,
     slope_score[inside_points.row.astype(int), inside_points.col.astype(int)] >= 0.45],
    ["lowland", "ridge", "slope"], default="other"
)
terrain_frequency = pd.Series(sample_terrain).value_counts(normalize=True)
candidates["terrain_diversity_score"] = 1 - candidates.terrain_type.map(terrain_frequency).fillna(0)

candidates["priority_score"] = 100 * (
    CONFIG.risk_weight*candidates.risk_score +
    CONFIG.uncertainty_weight*candidates.uncertainty_score +
    CONFIG.sparsity_weight*candidates.sparsity_score +
    CONFIG.recent_change_weight*candidates.recent_change_score +
    CONFIG.control_weight*candidates.control_score +
    CONFIG.terrain_diversity_weight*candidates.terrain_diversity_score
)

candidates["objective"] = np.select(
    [(candidates.control_score >= 0.45) & (candidates.risk_score <= 0.35),
     candidates.risk_score >= 0.65,
     candidates.uncertainty_score >= 0.65,
     candidates.recent_change_score >= 0.65],
    ["control", "hotspot_validation", "uncertainty_reduction", "change_validation"],
    default="coverage_gap"
)
candidates["access_status"] = "field_verification_required"
candidates.insert(0, "candidate_id", [f"ECOS-{i:05d}" for i in range(1, len(candidates)+1)])

print(f"Generated {len(candidates):,} candidates from the real raster grid.")
display(candidates.head())


## 5. Select a scientifically balanced field plan

The selector reserves space for controls, hotspots, and uncertainty checks when qualifying candidates exist, then fills remaining positions by overall priority. All selected sites must satisfy the minimum geographic spacing.


In [ ]:
CONTRIBUTION_LABELS = {
    "risk": "elevated EcoSignal risk",
    "uncertainty": "high model uncertainty",
    "sparsity": "low existing sample density",
    "recent_change": "strong recent elevation change",
    "control": "need for a low-risk control",
    "terrain_diversity": "underrepresented terrain type",
}


def add_explanation(row):
    values = {
        "risk": CONFIG.risk_weight*row.risk_score,
        "uncertainty": CONFIG.uncertainty_weight*row.uncertainty_score,
        "sparsity": CONFIG.sparsity_weight*row.sparsity_score,
        "recent_change": CONFIG.recent_change_weight*row.recent_change_score,
        "control": CONFIG.control_weight*row.control_score,
        "terrain_diversity": CONFIG.terrain_diversity_weight*row.terrain_diversity_score,
    }
    top = sorted(values, key=values.get, reverse=True)[:3]
    reason = "; ".join(CONTRIBUTION_LABELS[k] for k in top).capitalize() + "."
    actions = {
        "control": "Collect a low-exposure control sample.",
        "hotspot_validation": "Collect a targeted sample to validate this predicted hotspot.",
        "uncertainty_reduction": "Collect a verification sample to reduce model uncertainty.",
        "change_validation": "Collect a sample to validate the recent-change signal.",
        "coverage_gap": "Collect a sample to improve spatial and terrain coverage.",
    }
    return actions[row.objective], reason


def far_enough(row, selected, minimum_m):
    if not selected:
        return True
    distances = [math.hypot(row.x-s.x, row.y-s.y) for s in selected]
    return min(distances) >= minimum_m


def choose_from(pool, count, selected):
    for row in pool.sort_values("priority_score", ascending=False).itertuples():
        if len(selected) >= CONFIG.recommendations or count <= 0:
            break
        if row.candidate_id in {s.candidate_id for s in selected}:
            continue
        if far_enough(row, selected, CONFIG.minimum_selected_spacing_m):
            selected.append(row)
            count -= 1
    return selected


selected_rows = []
quota_plan = [
    ("control", CONFIG.minimum_controls),
    ("hotspot_validation", CONFIG.minimum_hotspots),
    ("uncertainty_reduction", CONFIG.minimum_uncertainty_checks),
]
for objective, quota in quota_plan:
    selected_rows = choose_from(candidates[candidates.objective == objective], quota, selected_rows)
selected_rows = choose_from(candidates, CONFIG.recommendations-len(selected_rows), selected_rows)

recommendations = pd.DataFrame([r._asdict() for r in selected_rows])
if recommendations.empty:
    raise RuntimeError("No candidate sites could be selected. Reduce spacing or inspect the valid-data mask.")
recommendations = recommendations.sort_values("priority_score", ascending=False).reset_index(drop=True)
recommendations.insert(0, "rank", np.arange(1, len(recommendations)+1))
explanations = recommendations.apply(add_explanation, axis=1, result_type='expand')
recommendations["recommended_action"] = explanations[0]
recommendations["reason"] = explanations[1]

show_columns = ["rank", "candidate_id", "objective", "terrain_type", "priority_score",
                "risk_score", "uncertainty_score", "distance_to_nearest_sample_m",
                "latitude", "longitude", "recommended_action", "reason", "access_status"]
display(recommendations[show_columns].round(4))


## 6. Visual review on the real risk map

Blue stars are the recommended next samples. Existing observations are white × markers. Candidate colors come from the real aligned EcoSignal risk raster.


In [ ]:
left, bottom, right, top = rasterio.transform.array_bounds(height, width, transform_2022)
fig, ax = plt.subplots(figsize=(12, 9))
im = ax.imshow(ecosignal_risk, cmap='inferno', vmin=0, vmax=1,
               extent=[left, right, bottom, top], origin='upper')
ax.scatter(inside_points.x, inside_points.y, marker='x', s=65, color='white',
           linewidth=1.5, label='Existing lichen samples')
ax.scatter(recommendations.x, recommendations.y, marker='*', s=250,
           color='#27a7ff', edgecolor='white', linewidth=1.0, label='Recommended next samples')
for r in recommendations.itertuples():
    ax.annotate(str(r.rank), (r.x, r.y), xytext=(7, 7), textcoords='offset points',
                color='white', weight='bold', fontsize=10)
plt.colorbar(im, ax=ax, label='EcoSignal risk')
ax.set(title='EcoSignal Active Sampling Plan — Real GPS-Aligned Data',
       xlabel=f'X ({profile_2022["crs"]})', ylabel=f'Y ({profile_2022["crs"]})')
ax.legend(loc='best')
plt.show()


## 7. Scientific QA

These tests run on the real outputs. They check bounds, coordinate validity, uniqueness, score ranges, exclusion around existing samples, and spacing between selected sites.


In [ ]:
def run_qa():
    assert len(recommendations) <= CONFIG.recommendations
    assert recommendations.candidate_id.is_unique
    assert recommendations.priority_score.between(0, 100).all()
    assert recommendations.risk_score.between(0, 1).all()
    assert recommendations.uncertainty_score.between(0, 1).all()
    assert recommendations.latitude.between(-90, 90).all()
    assert recommendations.longitude.between(-180, 180).all()
    assert (recommendations.distance_to_nearest_sample_m >= CONFIG.exclude_near_sample_m-1e-6).all()
    minimum_pair_spacing = None
    if len(recommendations) > 1:
        xy = recommendations[["x", "y"]].to_numpy(float)
        delta = xy[:, None, :] - xy[None, :, :]
        distances = np.sqrt((delta**2).sum(axis=2))
        np.fill_diagonal(distances, np.inf)
        minimum_pair_spacing = float(distances.min())
        assert minimum_pair_spacing >= CONFIG.minimum_selected_spacing_m-1e-6
    assert recommendations.reason.str.len().gt(20).all()
    return {
        "status": "PASS", "real_lichen_observations": int(len(inside_points)),
        "candidate_sites": int(len(candidates)), "selected_sites": int(len(recommendations)),
        "minimum_pair_spacing_m": None if minimum_pair_spacing is None else round(minimum_pair_spacing, 2),
        "difference_correlation": round(float(diff_correlation), 4),
        "objectives_selected": recommendations.objective.value_counts().to_dict(),
    }

qa_results = run_qa()
print(json.dumps(qa_results, indent=2))


## 8. Export for EcoSignal

The CSV is analysis-friendly. The GeoJSON is frontend/map-ready and includes the explanation, objective, scores, and the warning that access still requires field verification.


In [ ]:
OUTPUT_DIR = DEM_DIR / "ActiveSamplingPlanner"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

csv_path = OUTPUT_DIR / "active_sampling_recommendations.csv"
geojson_path = OUTPUT_DIR / "active_sampling_recommendations.geojson"
qa_path = OUTPUT_DIR / "active_sampling_qa.json"
recommendations.to_csv(csv_path, index=False)

properties_to_export = [
    "rank", "candidate_id", "objective", "terrain_type", "priority_score",
    "risk_score", "uncertainty_score", "sparsity_score", "recent_change_score",
    "control_score", "terrain_diversity_score", "distance_to_nearest_sample_m",
    "recommended_action", "reason", "access_status"
]
features = []
for row in recommendations.itertuples():
    props = {}
    for key in properties_to_export:
        value = getattr(row, key)
        if isinstance(value, (np.integer, np.floating)):
            value = value.item()
        props[key] = value
    features.append({
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": [float(row.longitude), float(row.latitude)]},
        "properties": props,
    })

geojson = {
    "type": "FeatureCollection",
    "name": "EcoSignal Active Sampling Recommendations",
    "planner_version": "1.0.0",
    "uncertainty_type": "heuristic_not_calibrated",
    "features": features,
}
geojson_path.write_text(json.dumps(geojson, indent=2))
qa_path.write_text(json.dumps(qa_results, indent=2))

print("Saved real-data planner outputs:")
print("-", csv_path)
print("-", geojson_path)
print("-", qa_path)


## Next validation step

Before calling this feature field-ready:

1. Add a land-use/accessibility layer so candidates can be labeled roadside, forest, school, park, slope, or lowland and inaccessible/private sites can be excluded automatically.
2. Review the six priority weights with an environmental-science mentor.
3. Visit a small prospective set of recommended and non-recommended sites, then test whether recommendations reduce map uncertainty or improve validation accuracy.
4. Replace the uncertainty heuristic with calibrated ensemble or cross-validation uncertainty when repeated spatial training data becomes available.
